# DisEnergy Interactive Demo: 3D Manifold, Box-Free Mask & Cubical Topology
Verify the zero-parameter DisEnergy operator, 3D elevation manifolds, crest-snapping watershed mask, and H0/H1 persistence diagrams on a synthetic textured phantom.

In [ ]:
import numpy as np
from disenergy import DisEnergyConfig, DisEnergyOperator, extract_box_free_mask, extract_cubical_persistence

h, w = 128, 128
yy, xx = np.mgrid[0:h, 0:w]
gt = ((yy - 64)**2 + (xx - 64)**2) <= 28**2
img = np.full((h, w, 3), 0.15, dtype=np.float32)
img[gt] = [0.85, 0.55, 0.35]

op = DisEnergyOperator(DisEnergyConfig(block_sizes=(2, 8, 32)))
out = op(img)
fused = out.fused_manifold.squeeze().numpy()
fine = out.scale_manifolds[2].squeeze().numpy()
mask = extract_box_free_mask(fused, fine_manifold=fine, method='watershed_filled')
tda = extract_cubical_persistence(fused)

dice = 2 * np.logical_and(mask, gt).sum() / (mask.sum() + gt.sum())
print(f'Dice={dice:.4f}, H0={len(tda.diagrams[0])}, H1={len(tda.diagrams[1])}, TDA dim={len(tda.feature_vector)}')